# Day 4 — Extra exercises: joint and hierarchical models

**For two kinds of reader.** If you **missed Day 4** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 4 contains** (see `../day-4-makeup.md`):

- **A1–A3 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-4-exercises-morning.ipynb` and `../day-4-exercises-afternoon.ipynb`.

The reflection prompts are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: a normal model with two unknowns, written in Stan; joint, marginal and conditional posteriors; an average versus a single new measurement. Part B: complete, no and partial pooling for eight wells; writing a hierarchical Stan model; shrinkage; predicting for a new well.

**Background.** Read the Day 4 slides first: `../day-4-slides-morning.pdf` (two unknowns) and `../day-4-slides-afternoon.pdf` (hierarchical models). Each task below names the slides it builds on. The demos in `../demos/` show the same steps on the slides' example.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel, from this `extra/` folder. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements.

**Stan.** Every fit runs live. `models/normal-one.stan` and `models/hier-wells.stan` are **templates**: complete the lines marked `TODO` (open them in JupyterLab's text editor) before fitting. The other model files are complete. If your installation fails, use UCloud (see the course page) and tell the lecturer early rather than at the deadline.

**Submitting (make-up).** Submit this notebook with outputs, the `.stan` files you edited, and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
DATA <- "data"; MODELS <- "models"     # run from the extra/ folder

## Part A — Nitrate in one well

The EU drinking-water limit for nitrate is **50 mg/L**. A farmer's well was sampled **12 times** over a year (`data/nitrate-one-well.csv`).

**The model** (*morning slide "A model with two unknowns"*): the samples are independent, yᵢ | μ, σ ~ Normal(μ, σ).

- μ = the well's long-run mean concentration (mg/L), prior μ ~ Normal(45, 15);
- σ = the spread from sample to sample (mg/L), prior σ ~ Normal⁺(0, 10) (half-normal).

### A1. Write and fit the model (25 min)

*Background: morning slides "The normal model in Stan" and "The fit: P04's posterior"; Day 2 for simulating from priors.*

(a) Open `models/normal-one.stan` and complete the `TODO`s: declare the parameters, add the likelihood, and generate one new sample `y_new`.
(b) Before fitting, simulate from the priors in two steps: draw μ and σ from their priors, then one sample y ~ Normal(μ, σ). What range of concentrations do the priors allow? Is the share of impossible negative values worrying?
(c) Fit the model (4 chains, `seed = 2026`), check the diagnostics, and report 90% intervals for μ and σ with units.

In [ ]:
one <- read_csv(file.path(DATA, "nitrate-one-well.csv"), show_col_types = FALSE)
y <- one$nitrate_mgL
c(n = length(y), mean = mean(y), sd = sd(y))

# (b) prior simulation
# mu_prior <- rnorm(4000, 45, 15);  sigma_prior <- abs(rnorm(4000, 0, 10))
# y_prior <- rnorm(4000, mu_prior, sigma_prior)
# quantile(y_prior, c(0.05, 0.5, 0.95));  mean(y_prior < 0)

# (c) mod_one <- cmdstan_model(file.path(MODELS, "normal-one.stan"))
#     fit_one <- mod_one$sample(data = list(N = length(y), y = y), chains = 4, seed = 2026, refresh = 0)
#     fit_one$summary(c("mu", "sigma", "y_new"), "mean", ~quantile(.x, c(0.05, 0.95)), "rhat", "ess_bulk", "ess_tail")
#     fit_one$diagnostic_summary()

<details><summary><b>Hint</b> (click to open)</summary>

Parameters: `real mu;` and `real<lower=0> sigma;`. Likelihood: `y ~ normal(mu, sigma);`. Generated: `real y_new = normal_rng(mu, sigma);`. The `<lower=0>` bound together with `sigma ~ normal(0, 10);` gives a half-normal prior.

</details>

**Your answer:** *(write here)*

### A2. Joint, marginal and conditional posteriors (20 min)

*Background: morning slides "Joint, marginal and conditional", "Conditional is not marginal" and "With draws, marginalising is free".*

(a) Plot the posterior draws of μ against σ (the joint posterior).
(b) Plot a histogram of each (the marginals).
(c) Explain in words why taking only the μ column of the draws gives the marginal posterior of μ: what happens to σ?
(d) Approximate two conditionals: keep the draws where σ is in its lowest third, and those where it is in its highest third. Compare the 90% intervals of μ in the two groups with the marginal interval. What does this show?

In [ ]:
# draws_one <- fit_one$draws(format = "df")     # one row per draw: mu, sigma, y_new
# (a) ggplot(draws_one, aes(x = mu, y = sigma)) + geom_point(alpha = 0.2)
# (d) low  <- draws_one |> filter(sigma < quantile(sigma, 1/3))
#     high <- draws_one |> filter(sigma > quantile(sigma, 2/3))
#     quantile(low$mu, c(0.05, 0.95))

<details><summary><b>Hint</b> (click to open)</summary>

Each row of the draws is one joint draw (μ, σ). Looking only at the μ column ignores which σ each draw came with: that is the same as adding up over σ, as on the grid slide.

</details>

**Your answer:** *(write here)*

### A3. Is the well above the limit? (20 min)

*Background: morning slides "Her average versus her next attempt" and "Questions the draws answer".*

(a) Compute P(μ > 50 | data) and P(y_new > 50 | data), and the 90% intervals of μ and `y_new`.
(b) Explain why the two probabilities differ so much.
(c) Which one answers the regulator's question *"is this well's long-run mean above the limit?"*, and which answers the family's question *"if we test the tap tomorrow, will the result exceed 50?"*

In [ ]:
# mean(draws_one$mu > 50);  mean(draws_one$y_new > 50)
# quantile(draws_one$mu, c(0.05, 0.95));  quantile(draws_one$y_new, c(0.05, 0.95))

<details><summary><b>Hint</b> (click to open)</summary>

y_new adds the sample-to-sample variation σ on top of the uncertainty about μ. Compare the two 90% intervals.

</details>

**Your answer:** *(write here)*

### Reflection A (optional)

In 80–120 words: explain the difference between a **marginal** and a **conditional** posterior using your A2 result, and say why P(μ > 50) and P(y_new > 50) differ.

**Your answer:** *(write here)*

## Part B — Eight wells in one catchment

The water authority sampled **eight wells** in one catchment with very unequal effort (`data/nitrate-eight-wells.csv`): from 15 samples (W1) down to a single sample (W8). The questions: *which wells are likely to have a mean above 50 mg/L, and what should we expect for a well that has not been sampled yet?*

### B1. Complete pooling and no pooling (20 min)

*Background: afternoon slides "Attempts are grouped within people" and "Three ways to use the class" (here: samples within wells).*

(a) Tabulate n, mean and SD per well.
(b) Read and fit the supplied `models/complete-pool.stan` and `models/no-pool.stan`.
(c) For each well, report the posterior mean of θⱼ (the well's mean) under both models. What does each model assume about how wells differ? Why is the no-pooling estimate for W8 unreliable?

In [ ]:
wells <- read_csv(file.path(DATA, "nitrate-eight-wells.csv"), show_col_types = FALSE)
well_names <- sort(unique(wells$well))
wells <- wells |> mutate(id = match(well, well_names))          # Stan needs wells as 1, ..., J
wdat <- list(N = nrow(wells), J = length(well_names), well = wells$id, y = wells$nitrate_mgL)

# (a) wells |> group_by(well) |> summarise(n = n(), mean = mean(nitrate_mgL), sd = sd(nitrate_mgL))
# (b) fit_cp <- cmdstan_model(file.path(MODELS, "complete-pool.stan"))$sample(data = wdat, chains = 4, seed = 2026, refresh = 0)
#     fit_np <- cmdstan_model(file.path(MODELS, "no-pool.stan"))$sample(data = wdat, chains = 4, seed = 2026, refresh = 0)
# (c) fit_np$summary("theta", "mean")

<details><summary><b>Hint</b> (click to open)</summary>

Complete pooling: one mean for all wells. No pooling: eight unrelated means, each with its own Normal(45, 15) prior, and a shared within-well SD σ. W8 has one sample, so its no-pooling estimate rests on that single reading (and the prior).

</details>

**Your answer:** *(write here)*

### B2. Write the hierarchical model (30 min)

*Background: afternoon slides "The hierarchical model", "The hierarchical model in Stan" and "Non-centred: same model, easier shape".*

Complete `models/hier-wells.stan`: well means θⱼ ~ Normal(μ, τ), written in the **non-centred** form θⱼ = μ + τ·zⱼ with zⱼ ~ Normal(0, 1). In `generated quantities`, add `theta_new` (the mean of a **new** well, θ_new ~ Normal(μ, τ)) and `y_new_well` (one sample from it).

(a) Fit it (4 chains, `seed = 2026`) and check the diagnostics, including divergences.
(b) Report 90% intervals for μ, τ and σ. Interpret τ against σ: which is larger, and what does that mean for these wells?

In [ ]:
# mod_h <- cmdstan_model(file.path(MODELS, "hier-wells.stan"))
# fit_h <- mod_h$sample(data = wdat, chains = 4, seed = 2026, refresh = 0)
# fit_h$diagnostic_summary()
# a few divergences? refit with smaller steps:
# fit_h <- mod_h$sample(data = wdat, chains = 4, seed = 2026, refresh = 0, adapt_delta = 0.95)
# fit_h$summary(c("mu", "tau", "sigma", "theta_new"), "mean", ~quantile(.x, c(0.05, 0.95)), "rhat", "ess_bulk")

<details><summary><b>Hint</b> (click to open)</summary>

`vector[J] theta = mu + tau * z;` in transformed parameters; `z ~ std_normal();` and `y ~ normal(theta[well], sigma);` in the model; `real theta_new = normal_rng(mu, tau);` and `real y_new_well = normal_rng(theta_new, sigma);` in generated quantities. If there are a few divergences, try `adapt_delta = 0.95` in `$sample()` (Day 3 slide on divergences).

</details>

**Your answer:** *(write here)*

### B3. Shrinkage and a new well (25 min)

*Background: afternoon slides "Two sources of information about θⱼ", "Where the weight w comes from", "Shrinkage in the class" and "Predicting: a person we know, or a new person".*

(a) Add the partial-pooling estimates to your B1 table and plot, for each well, an arrow from its own mean to the partial-pooling estimate, with a line at μ. Which wells move most, and why?
(b) For W8 (one sample), compute the weight w = (n/σ²) / (n/σ² + 1/τ²) with the posterior means of σ and τ, and check w·ȳ + (1 − w)·μ against the fitted estimate.
(c) Report P(θⱼ > 50 | data) for each well under no pooling and partial pooling.
(d) Give the 90% interval of `theta_new` and P(θ_new > 50). Why is this interval wider than for any sampled well?

In [ ]:
# (a) per_well <- per_well |> mutate(partial = fit_h$summary("theta", "mean")$mean)
#     mu_hat <- fit_h$summary("mu")$mean
#     ggplot(per_well, aes(y = well)) +
#       geom_vline(xintercept = mu_hat, colour = "orange") +
#       geom_segment(aes(x = own_mean, xend = partial, yend = well), arrow = arrow(length = unit(0.1, "inches"))) +
#       geom_point(aes(x = own_mean, colour = "own mean"), size = 3) +
#       geom_point(aes(x = partial, colour = "partial pooling"), size = 3) +
#       scale_colour_manual(values = c("own mean" = "red", "partial pooling" = "navy"), name = NULL)
# (c) draws_h <- fit_h$draws(format = "df");  mean(draws_h$`theta[8]` > 50)
# (d) quantile(draws_h$theta_new, c(0.05, 0.95));  mean(draws_h$theta_new > 50)

<details><summary><b>Hint</b> (click to open)</summary>

How far a well moves depends on how much data it has (nⱼ/σ²) relative to how different wells tend to be (1/τ²), and on how far its own mean is from μ. A new well has no data of its own: its mean is uncertain by about τ, plus the uncertainty about μ.

</details>

**Your answer:** *(write here)*

### Reflection B (optional)

In 80–120 words: what does partial pooling share between wells, and why does the new-well prediction depend on τ? Name one reason the eight wells might **not** be exchangeable (*afternoon slide "Why one common distribution? Exchangeability"*).

**Your answer:** *(write here)*

## C. Optional challenges

**C1. Centred versus non-centred** (*afternoon slides "A warning sign: the funnel" and "Non-centred: same model, easier shape"*). Fit the supplied `models/hier-wells-centered.stan` and your `hier-wells.stan`, both with default settings (no `adapt_delta`). Compare divergences and the bulk ESS of τ. Plot log τ against θ₈ − μ for the centred fit, with divergent draws highlighted (`fit$sampler_diagnostics(format = "df")$divergent__`).

**C2. The prior on τ.** Refit the hierarchical model with τ ~ Normal⁺(0, 5) and with τ ~ Normal⁺(0, 30) (edit a copy of the file). How much do W8's estimate and P(θ_new > 50) change? Why are eight wells too few to learn τ precisely?

In [ ]:
# Optional challenge work

**Your answer:** *(write here)*

## Make-up checklist

- [ ] A1–A3 and B1–B3 · [ ] in-class exercises (`../day-4-exercises-morning.ipynb` and `../day-4-exercises-afternoon.ipynb`)
- [ ] Completed `models/normal-one.stan` and `models/hier-wells.stan` submitted with the notebook
- [ ] Diagnostics reported before interpretation; the shrinkage plot has labelled axes and units
- [ ] The notebook runs from a fresh kernel